In [0]:
# ============================================================
# NOTEBOOK 4 — DELTA LAKE DEEP DIVE
# Topics : MERGE INTO, SCD Type 2, Schema Evolution,
#           Time Travel, RESTORE, Change Data Feed
# ============================================================

CATALOG       = "brazilian-ecommerce"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
DELTA_SCHEMA  = "delta_demos"   # separate schema for clean demos

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{DELTA_SCHEMA}`")

print(f"Demo schema ready : {CATALOG}.{DELTA_SCHEMA}")

In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, to_timestamp,
    when, sha2, concat_ws
)

# We'll simulate incremental loads on a smaller orders subset
# Pull delivered orders from Silver as our "Day 1" snapshot

df_day1 = (
    spark.table(f"`{CATALOG}`.`{SILVER_SCHEMA}`.`orders_master`")
    .filter(col("order_status") == "DELIVERED")
    .select(
        "order_id",
        "customer_unique_id",
        "customer_state",
        "product_category_name",
        "price",
        "line_total",
        "is_late_delivery",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    )
    .dropDuplicates(["order_id"])
    .limit(5000)   # manageable size for demo
)

ORDERS_TABLE = f"`{CATALOG}`.`{DELTA_SCHEMA}`.`orders_incremental`"

# Write Day 1 snapshot
(df_day1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ORDERS_TABLE)
)

cnt = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE}").collect()[0]["c"]
print(f"Day 1 snapshot written : {cnt:,} rows")

In [0]:
from pyspark.sql import Row
import random

# ---- Simulate 3 types of CDC events ----

# 1. NEW orders arriving on Day 2
new_orders = spark.createDataFrame([
    Row(order_id="NEW_ORD_001", customer_unique_id="CUST_A",
        customer_state="SP", product_category_name="electronics",
        price=499.99, line_total=529.99,
        is_late_delivery=False,
        order_purchase_timestamp="2018-09-01 10:00:00",
        order_delivered_customer_date="2018-09-05 14:00:00",
        _change_type="INSERT"),

    Row(order_id="NEW_ORD_002", customer_unique_id="CUST_B",
        customer_state="RJ", product_category_name="furniture",
        price=1200.00, line_total=1250.00,
        is_late_delivery=True,
        order_purchase_timestamp="2018-09-01 11:30:00",
        order_delivered_customer_date="2018-09-10 09:00:00",
        _change_type="INSERT"),
])

# Cast timestamp strings
new_orders = new_orders \
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date")))

# 2. UPDATED orders — price corrected on existing orders
# Pull 3 existing order_ids from Day 1
existing_ids = (spark.sql(f"SELECT order_id FROM {ORDERS_TABLE} LIMIT 3")
                .collect())
updated_orders = (
    spark.sql(f"""
        SELECT order_id, customer_unique_id, customer_state,
               product_category_name,
               ROUND(price * 1.1, 2)       AS price,
               ROUND(line_total * 1.1, 2)  AS line_total,
               is_late_delivery,
               order_purchase_timestamp,
               order_delivered_customer_date
        FROM {ORDERS_TABLE}
        LIMIT 3
    """)
    .withColumn("_change_type", lit("UPDATE"))
)

# 3. Combine into one incoming batch (the "delta" arriving on Day 2)
from functools import reduce
from pyspark.sql import DataFrame

df_incoming = new_orders.drop("_change_type") \
    .union(updated_orders.drop("_change_type"))

df_incoming.createOrReplaceTempView("incoming_day2")
print(f"Incoming Day 2 batch : {df_incoming.count()} rows")
print(f"  - New orders  : {new_orders.count()}")
print(f"  - Updated     : {updated_orders.count()}")

In [0]:
# SCD Type 1 = simply overwrite changed columns, no history kept
# Use when you only care about current state (e.g. current price)

spark.sql(f"""
    MERGE INTO {ORDERS_TABLE} AS target
    USING incoming_day2 AS source
    ON target.order_id = source.order_id

    -- If order exists and price changed → UPDATE it (Type 1: overwrite)
    WHEN MATCHED AND target.price <> source.price THEN
        UPDATE SET
            target.price                        = source.price,
            target.line_total                   = source.line_total,
            target.is_late_delivery             = source.is_late_delivery

    -- If order doesn't exist in target → INSERT it
    WHEN NOT MATCHED BY TARGET THEN
        INSERT (
            order_id, customer_unique_id, customer_state,
            product_category_name, price, line_total,
            is_late_delivery, order_purchase_timestamp,
            order_delivered_customer_date
        )
        VALUES (
            source.order_id, source.customer_unique_id, source.customer_state,
            source.product_category_name, source.price, source.line_total,
            source.is_late_delivery, source.order_purchase_timestamp,
            source.order_delivered_customer_date
        )
""")

print("MERGE INTO (SCD Type 1) complete.")
cnt = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE}").collect()[0]["c"]
print(f"Orders table after merge : {cnt:,} rows")

# Verify the merge — new orders should be present
spark.sql(f"""
    SELECT order_id, price, line_total
    FROM {ORDERS_TABLE}
    WHERE order_id IN ('NEW_ORD_001', 'NEW_ORD_002')
""").show()

In [0]:
# SCD Type 2 = keep ALL historical versions of a record
# New version gets inserted, old version gets closed with end_date
# Used for: customer address history, product price history, status changes

# Create a dedicated SCD2 table for customer segments
SCD2_TABLE = f"`{CATALOG}`.`{DELTA_SCHEMA}`.`customer_segments_scd2`"

# Build initial customer segment snapshot (Day 1)
df_scd2_day1 = (
    spark.sql(f"""
        SELECT
            customer_unique_id,
            customer_state,
            SUM(line_total) AS lifetime_revenue
        FROM `{CATALOG}`.`{SILVER_SCHEMA}`.`orders_master`
        WHERE order_status = 'DELIVERED'
        GROUP BY customer_unique_id, customer_state
    """)
    .withColumn("customer_segment",
        when(col("lifetime_revenue") >= 1000, lit("high_value"))
        .when(col("lifetime_revenue") >= 300,  lit("mid_value"))
        .otherwise(lit("low_value")))
    # SCD2 columns
    .withColumn("effective_start_date", to_timestamp(lit("2018-01-01 00:00:00")))
    .withColumn("effective_end_date",   lit(None).cast("timestamp"))
    .withColumn("is_current",           lit(True))
    .withColumn("record_hash",
                sha2(concat_ws("|",
                    col("customer_segment"),
                    col("customer_state")
                ), 256))
    .limit(1000)
)

(df_scd2_day1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SCD2_TABLE)
)

print(f"SCD2 Day 1 written : {df_scd2_day1.count():,} rows")
display(spark.sql(f"SELECT * FROM {SCD2_TABLE} LIMIT 5"))

In [0]:
# Simulate Day 2: some customers upgraded to higher segment
# In production this comes from your incremental pipeline run

from pyspark.sql.functions import current_date, current_timestamp

# Grab 5 mid_value customers and upgrade them to high_value
df_upgrades = (
    spark.sql(f"""
        SELECT customer_unique_id, customer_state,
               lifetime_revenue * 1.5 AS lifetime_revenue
        FROM {SCD2_TABLE}
        WHERE customer_segment = 'mid_value'
        AND is_current = true
        LIMIT 5
    """)
    .withColumn("customer_segment", lit("high_value"))
    .withColumn("record_hash",
                sha2(concat_ws("|",
                    lit("high_value"),
                    col("customer_state")
                ), 256))
)

df_upgrades.createOrReplaceTempView("segment_updates_day2")
print(f"Customers to upgrade : {df_upgrades.count()}")

# SCD Type 2 MERGE:
# Step 1 — Close the old record (set end_date, is_current = false)
# Step 2 — Insert the new record (new segment, new start_date)

spark.sql(f"""
    MERGE INTO {SCD2_TABLE} AS target
    USING segment_updates_day2 AS source
    ON  target.customer_unique_id = source.customer_unique_id
    AND target.is_current = true
    AND target.record_hash <> source.record_hash

    -- Close old record
    WHEN MATCHED THEN
        UPDATE SET
            target.effective_end_date = current_timestamp(),
            target.is_current         = false
""")

# Insert new versions as separate records
(df_upgrades
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date",   lit(None).cast("timestamp"))
    .withColumn("is_current",           lit(True))
    .write.format("delta")
    .mode("append")
    .saveAsTable(SCD2_TABLE)
)

total = spark.sql(f"SELECT COUNT(*) as c FROM {SCD2_TABLE}").collect()[0]["c"]
current = spark.sql(f"SELECT COUNT(*) as c FROM {SCD2_TABLE} WHERE is_current=true").collect()[0]["c"]
history = total - current
print(f"\nSCD2 after Day 2 update:")
print(f"  Total records   : {total:,}")
print(f"  Current records : {current:,}")
print(f"  History records : {history:,}")

In [0]:
# Time travel is Delta's most powerful feature for production debugging

# Check full history of our orders table
print("=== Transaction History ===")
spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}") \
     .select("version", "timestamp", "operation", "operationMetrics") \
     .show(10, truncate=False)

In [0]:
# Read a specific version — what did the table look like before the merge?
df_before_merge = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)     # version 0 = original Day 1 load
    .table(f"`{CATALOG}`.`{DELTA_SCHEMA}`.`orders_incremental`")
)

df_current = spark.table(ORDERS_TABLE)

v0_count  = df_before_merge.count()
cur_count = df_current.count()

print(f"Version 0 (before merge) : {v0_count:,} rows")
print(f"Current version          : {cur_count:,} rows")
print(f"Net new rows from merge  : {cur_count - v0_count:,}")

In [0]:
# Read by timestamp instead of version number
# (more practical in production — you know WHEN the bad load ran)
from pyspark.sql.functions import current_timestamp as now

# Get the timestamp of version 0 from history
v0_ts = spark.sql(f"""
    SELECT timestamp FROM (
        DESCRIBE HISTORY {ORDERS_TABLE}
    ) WHERE version = 0
""").collect()[0]["timestamp"]

print(f"Version 0 timestamp : {v0_ts}")

df_by_ts = (
    spark.read
    .format("delta")
    .option("timestampAsOf", str(v0_ts))
    .table(f"`{CATALOG}`.`{DELTA_SCHEMA}`.`orders_incremental`")
)
print(f"Rows at that timestamp : {df_by_ts.count():,}")

In [0]:
# RESTORE — roll back the table to a previous version IN PLACE
# This is the production recovery command — not just a read

# Simulate a bad write first
spark.sql(f"""
    INSERT INTO {ORDERS_TABLE}
    SELECT order_id || '_BAD', customer_unique_id, customer_state,
           product_category_name, -999.0, -999.0,
           is_late_delivery, order_purchase_timestamp,
           order_delivered_customer_date
    FROM {ORDERS_TABLE}
    LIMIT 10
""")

bad_count = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE}").collect()[0]["c"]
bad_price = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE} WHERE price = -999.0").collect()[0]["c"]
print(f"After bad write  : {bad_count:,} rows, {bad_price} rows with price=-999")

# Now RESTORE to the version just before the bad write
latest_good_version = spark.sql(f"""
    SELECT MAX(version) - 1 as v FROM (DESCRIBE HISTORY {ORDERS_TABLE})
""").collect()[0]["v"]

spark.sql(f"""
    RESTORE TABLE {ORDERS_TABLE}
    TO VERSION AS OF {latest_good_version}
""")

restored_count = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE}").collect()[0]["c"]
bad_after = spark.sql(f"SELECT COUNT(*) as c FROM {ORDERS_TABLE} WHERE price = -999.0").collect()[0]["c"]
print(f"\nAfter RESTORE    : {restored_count:,} rows, {bad_after} rows with price=-999")
print("Table successfully restored!")

In [0]:
# Schema evolution = safely adding columns to a Delta table
# without breaking existing readers

SCHEMA_TABLE = f"`{CATALOG}`.`{DELTA_SCHEMA}`.`schema_evolution_demo`"

# --- Version 1: original schema ---
df_v1 = spark.sql(f"""
    SELECT order_id, customer_unique_id, price
    FROM {ORDERS_TABLE}
    LIMIT 100
""")

(df_v1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SCHEMA_TABLE)
)
print("V1 schema:")
spark.sql(f"DESCRIBE {SCHEMA_TABLE}").show()

In [0]:
from pyspark.sql.functions import lit, current_date

# --- Version 2: add new columns (schema evolution) ---
df_v2 = spark.sql(f"""
    SELECT order_id, customer_unique_id, price,
           customer_state,            -- new column
           product_category_name      -- new column
    FROM {ORDERS_TABLE}
    LIMIT 50
""")

# mergeSchema = True allows adding new columns without error
(df_v2.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")    # KEY option for schema evolution
    .saveAsTable(SCHEMA_TABLE)
)

print("\nV2 schema after evolution:")
spark.sql(f"DESCRIBE {SCHEMA_TABLE}").show()

# Rows from V1 will have NULL in the new columns — expected behaviour
spark.sql(f"""
    SELECT order_id, price, customer_state, product_category_name,
           CASE WHEN customer_state IS NULL THEN 'V1 row' ELSE 'V2 row' END AS row_version
    FROM {SCHEMA_TABLE}
    LIMIT 10
""").show()

In [0]:
# Change Data Feed = Delta can track row-level changes (insert/update/delete)
# as a readable stream or batch — like a built-in CDC log

CDF_TABLE = f"`{CATALOG}`.`{DELTA_SCHEMA}`.`cdf_demo`"

# Enable CDF on a table (can also be set at write time)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CDF_TABLE}
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
    AS SELECT order_id, customer_unique_id, price, customer_state
       FROM {ORDERS_TABLE}
       LIMIT 200
""")

print("CDF table created.")

# Make some changes to generate CDF entries
spark.sql(f"""
    UPDATE {CDF_TABLE}
    SET price = ROUND(price * 1.05, 2)
    WHERE customer_state = 'SP'
""")

spark.sql(f"""
    DELETE FROM {CDF_TABLE}
    WHERE price < 50
""")

spark.sql(f"""
    INSERT INTO {CDF_TABLE}
    VALUES ('CDF_NEW_001', 'TEST_CUST', 299.99, 'MG')
""")

# Now READ the change feed — see exactly what changed
df_changes = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table(CDF_TABLE)
)

print("\nChange feed entries by operation type:")
df_changes.groupBy("_change_type").count().show()

print("\nSample change feed rows:")
display(
    df_changes
    .select("order_id", "price", "customer_state",
            "_change_type", "_commit_version", "_commit_timestamp")
    .orderBy("_commit_version")
    .limit(15)
)

In [0]:
# Things interviewers love to ask about — table internals

print("=== DESCRIBE DETAIL ===")
spark.sql(f"DESCRIBE DETAIL {ORDERS_TABLE}").show(vertical=True)

In [0]:
print("=== TABLE PROPERTIES ===")
spark.sql(f"SHOW TBLPROPERTIES {ORDERS_TABLE}").show(truncate=False)

In [0]:
# Collect statistics for data skipping (important for query performance)
print("=== COMPUTE STATISTICS ===")
spark.sql(f"""
    ANALYZE TABLE {ORDERS_TABLE}
    COMPUTE STATISTICS FOR ALL COLUMNS
""")
print("Statistics computed — Delta will now use column-level min/max for data skipping.")

# Final history showing everything we did in this notebook
print("\n=== FULL TRANSACTION HISTORY ===")
spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}") \
     .select("version", "timestamp", "operation", "userName") \
     .show(20, truncate=False)